# Territorial Digital Divide – Geospatial Raster Analysis
**Cusco, Peru | NASA VNL × Mobile Coverage**

Pipeline measuring digital inequality using nighttime radiance and mobile network density as proxies for urbanization and internet access.

---
## Setup

In [ ]:
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.crs import CRS
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
from scipy.ndimage import gaussian_filter
from scipy import stats
import seaborn as sns
import pandas as pd
from pathlib import Path

# Paths
DATA_DIR   = Path('../data')
OUTPUT_DIR = Path('../output')
OUTPUT_DIR.mkdir(exist_ok=True)

VNL_PATH  = DATA_DIR / 'VNL_cusco_2025.tif'
KERN_PATH = DATA_DIR / 'kernel_cobmovil2019_50m.tif'

---
## Steps 0–1 | Load & Inspect Rasters

In [ ]:
def inspect_raster(path: Path, label: str) -> dict:
    """Open a raster and print its key metadata."""
    with rasterio.open(path) as src:
        info = {
            'label':      label,
            'crs':        src.crs.to_string(),
            'width':      src.width,
            'height':     src.height,
            'count':      src.count,
            'nodata':     src.nodata,
            'bounds':     src.bounds,
            'res_x':      src.res[0],
            'res_y':      src.res[1],
            'dtype':      src.dtypes[0],
        }
    print(f"\n{'='*55}")
    print(f"  {label}")
    print(f"{'='*55}")
    print(f"  CRS          : {info['crs']}")
    print(f"  Dimensions   : {info['height']} rows × {info['width']} cols")
    print(f"  Bands        : {info['count']}")
    print(f"  Data type    : {info['dtype']}")
    print(f"  NoData value : {info['nodata']}")
    print(f"  Bounds       : {info['bounds']}")
    print(f"  Pixel res    : {info['res_x']:.6f} × {info['res_y']:.6f} (x, y)")
    return info

vnl_info  = inspect_raster(VNL_PATH,  'VNL_cusco_2025.tif  (NASA Nighttime Radiance)')
kern_info = inspect_raster(KERN_PATH, 'kernel_cobmovil2019_50m.tif  (Mobile Coverage Kernel)')